<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Gemma_3_bfloat16_vs_float16_vs_float32%2C_evaluation_and_AMP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

More details in this article: [BF16 vs. FP16 vs. FP32 for Gemma 3 Inference — Mind Your Data Type](https://kaitchup.substack.com/p/mind-your-data-type-bf16-vs-fp16)


This notebook evaluates Gemma 3 using different data types: float16, float32, abd bfloat16. It also shows how to set AMP for inference with float16.

*Note: I compiled from source vLLM and Transformers to get full support for Gemma 3. This can be slow. Later, you will only need to install the packages transformers and vllm.*

# Installation

In [ ]:
!pip install --upgrade hf_transfer lm_eval auto-gptq optimum langdetect immutabledict
!git clone https://github.com/vllm-project/vllm.git && cd vllm && VLLM_USE_PRECOMPILED=1 pip install --editable .
!pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 30.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 90.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Download the Models

In [ ]:
models = [
          "google/gemma-3-4b-it",
          "google/gemma-3-12b-it",
          "google/gemma-3-27b-it",
          ]

for m in models:
    !HF_HUB_ENABLE_HF_TRANSFER=1 huggingface-cli download {m} --exclude *.pth

/root/.cache/huggingface/hub/models--google--gemma-3-4b-it/snapshots/dbd91bbaf64a0e591f4340ce8b66fd1dba9ab6bd
.gitattributes: 100%|██████████████████████| 1.97k/1.97k [00:00<00:00, 25.1MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--google--gemma-3-12b-it/blobs/3c4370fda8fab65837da0173e4e02cfa23e598b4
README.md: 100%|███████████████████████████| 25.2k/25.2k [00:00<00:00, 53.2MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--google--gemma-3-12b-it/blobs/a2002d5528efaa4788e1efea22243eec18ab5019
added_tokens.json: 100%|██████████████████████| 35.0/35.0 [00:00<00:00, 252kB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--google--gemma-3-12b-it/blobs/e17bde03d42feda32d1abfca6d3b598b9a020df7
chat_template.json: 100%|██████████████████| 1.61k/1.61k [00:00<00:00, 9.03MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--google--gemma-3-12b-it/blobs/719b0cd0d7a373a400b0c119ee0e051f41ea88d9
co

# Evaluation with IFEval

In [ ]:
for m in models:
    model_name = m
    for d in ["float16", "float32", "bfloat16"]:
      !lm_eval --model vllm \
      --model_args pretrained={model_name},dtype={d},max_model_len=12000 \
      --tasks leaderboard_ifeval\
      --device cuda:0 \
      --batch_size auto \
      --num_fewshot 0 \
      --output_path results.{d}

INFO 03-17 09:18:40 [__init__.py:256] Automatically detected platform cuda.
2025-03-17:09:18:48,446 INFO     [lm_eval.__main__:379] Selected Tasks: ['leaderboard_ifeval']
2025-03-17:09:18:48,449 INFO     [lm_eval.evaluator:169] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-03-17:09:18:48,449 INFO     [lm_eval.evaluator:206] Initializing vllm model, with arguments: {'pretrained': 'google/gemma-3-4b-it', 'dtype': 'float16', 'max_model_len': 12000}
INFO 03-17 09:18:49 [config.py:2579] Downcasting torch.float32 to torch.float16.
INFO 03-17 09:18:54 [config.py:583] This model supports multiple tasks: {'generate', 'classify', 'reward', 'embed', 'score'}. Defaulting to 'generate'.
INFO 03-17 09:18:54 [config.py:1677] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-17 09:18:57 [core.py:53] Initializing a V1 LLM engine (v0.8.0rc2.dev5+gcd0cd851) with config: model='google/gemma-3-4b-it', 

# Accurate Inference with Float16 Using AMP

In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from PIL import Image
import requests
import torch
import torch.nn as nn

def test_dtype(model_id, model_dtype, autocast=False):
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")



    model = Gemma3ForConditionalGeneration.from_pretrained(
        model_id, device_map="auto",torch_dtype=model_dtype
    ).eval()


    processor = AutoProcessor.from_pretrained(model_id)

    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."}]
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"},
                {"type": "text", "text": "Describe this image in detail."}
            ]
        }
    ]

    if autocast:
        with torch.cuda.amp.autocast(dtype=torch.float32):
            inputs = processor.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt"
                ).to(model.device)

            input_len = inputs["input_ids"].shape[-1]

            with torch.inference_mode():
                generation = model.generate(**inputs, max_new_tokens=100, do_sample=False)
                generation = generation[0][input_len:]

            decoded = processor.decode(generation, skip_special_tokens=True)
            print(decoded)

    else:
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt"
        ).to(model.device)

        input_len = inputs["input_ids"].shape[-1]

        with torch.inference_mode():
            generation = model.generate(**inputs, max_new_tokens=100, do_sample=False)
            generation = generation[0][input_len:]

        decoded = processor.decode(generation, skip_special_tokens=True)
        print(decoded)

    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

    print(f"Peak reserved memory = {used_memory} GB.")
    print("-----")

In [ ]:
test_dtype("google/gemma-3-4b-it", "bfloat16", autocast=False)

GPU = NVIDIA A100 80GB PCIe. Max memory = 79.256 GB.
0.0 GB of memory reserved.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Okay, here's a detailed description of the image:

**Overall Impression:**

The image is a close-up shot of a vibrant garden scene, focusing on a cluster of pink cosmos flowers and a busy bumblebee. It has a slightly soft, natural feel, likely captured in daylight.

**Foreground:**

*   **Cosmos Flowers:** The dominant feature is a large, bright pink cosmos flower in the center. It’s fully open, displaying its layered petals and a yellow
Peak reserved memory = 8.254 GB.
-----


In [ ]:
test_dtype("google/gemma-3-4b-it", "float32", autocast=False)

GPU = NVIDIA A100 80GB PCIe. Max memory = 79.256 GB.
0.0 GB of memory reserved.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Okay, here's a detailed description of the image:

**Overall Impression:**

The image is a close-up shot of a vibrant garden scene, focusing on a cluster of pink cosmos flowers and a busy bumblebee. It has a slightly soft, natural feel, likely captured in daylight.

**Foreground:**

*   **Cosmos Flowers:** The dominant feature is a large, bright pink cosmos flower in the center. It’s fully open, displaying its layered petals and a yellow
Peak reserved memory = 16.426 GB.
-----


Empty output with float16:

In [ ]:
test_dtype("google/gemma-3-4b-it", "float16", autocast=False)

GPU = NVIDIA A100 80GB PCIe. Max memory = 79.256 GB.
0.0 GB of memory reserved.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



Peak reserved memory = 8.322 GB.
-----


In [ ]:
test_dtype("google/gemma-3-4b-it", "float16", autocast=True)

GPU = NVIDIA A100 80GB PCIe. Max memory = 79.256 GB.
0.0 GB of memory reserved.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/tmp/ipykernel_14621/2038362016.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Okay, here's a detailed description of the image:

**Overall Impression:**

The image is a close-up shot of a vibrant garden scene, focusing on a cluster of pink cosmos flowers and a busy bumblebee. It has a slightly soft, natural feel, likely captured in daylight.

**Foreground:**

*   **Cosmos Flowers:** The dominant feature is a large, bright pink cosmos flower in the center. It’s fully open, displaying its layered petals and a yellow
Peak reserved memory = 11.029 GB.
-----
